## Downloading all the dependencies

In [ ]:
!pip install -U langchain
!pip install -U langchain-community
!pip install -U langchain-classic
!pip install -U langchain-text-splitters
!pip install -U langchain-huggingface
!pip install -U langchain-chroma
!pip install -U chromadb
!pip install -U sentence-transformers
!pip install -U langchain-nvidia-ai-endpoints
!pip install rank_bm25

## Importing all the modules


In [ ]:
from langchain_classic.storage import LocalFileStore, create_kv_docstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever, BM25Retriever
from langchain_text_splitters import  RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage
from langchain_classic.schema import Document
from sentence_transformers import CrossEncoder
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import os, re, base64
import getpass
import textwrap
import json
import logging

## Initializing the Logger

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)

logger = logging.getLogger(__name__)

## Checking Cuda(Nvidia GPU) is available..


In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

## Defining the embedding Model

In [ ]:
local_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'}, # Use 'cuda' if you have an NVIDIA GPU
    encode_kwargs={'normalize_embeddings': True}
)


## Loading the VectorDb

In [ ]:
persist_dir = ""  # Set this to your desired Chroma persistence directory
parent_store_dir = "" # Set this to your desired Parent Storage directory

# Load the Persistent Parent and Child Storage 
fs = LocalFileStore(parent_store_dir)
store = create_kv_docstore(fs)

# Setup Chroma as before
vectorstore = Chroma(
    collection_name="med_rag_local",
    embedding_function=local_embeddings,
    persist_directory=persist_dir
)


child_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
# 4. Initialize Retriever with the PERSISTENT store
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter
)

## Initializing the llms

In [ ]:
if "NVIDIA_API_KEY" not in os.environ:
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA NIM API Key: ")

# 2. Initialize the Best Free LLM (Llama 3.3 70B Instruct)
print("🚀 Initializing Llama-3.3-70B via NVIDIA API...")
llm = ChatNVIDIA(
    model="meta/llama-3.3-70b-instruct",
    temperature=0.2, # Keep it low for factual textbook answers
    max_tokens=1024
)


print("🚀 Initializing Llama-3.2-90B-Vision via NVIDIA API...")
vision_llm = ChatNVIDIA(
    model="meta/llama-3.2-90b-vision-instruct",
    temperature=0.2,
    max_completion_tokens=1024
)

## Implementing HyDE (Hypothethical Document Embedding)

In [ ]:
def generate_hyde_vector(query: str, llm_client, embedding_model) -> list[float]:
    """
    Generates a hypothetical document and returns its vector embedding.

    Args:
        query: The user's search query.
        llm_client: Your initialized LLM (e.g., your Llama generator function).
        embedding_model: Your initialized BGE-M3 embedder.
    """

    # 1. The Prompt: Instruct the LLM to write a textbook-style answer
    prompt = f"""Given the following question, write a hypothetical, detailed textbook passage that directly answers it.
    Write in an academic tone, use standard formatting, and include relevant domain keywords that might appear in a textbook.

    Question: {query}

    Hypothetical Textbook Passage:"""

    # 2. Generate the hypothetical document
    # (Adjust the calling method based on your specific LLM wrapper)

    response = llm_client.invoke([
        HumanMessage(content=prompt)
    ])

    hypothetical_doc = response.content
    # print(f"--- Hypothetical Document Generated ---\n{hypothetical_doc}\n---------------------------------------")

    # 3. Embed the generated document instead of the raw query
    # (Adjust .encode() based on whether you are using HuggingFace, SentenceTransformers, etc.)
    hyde_vector = embedding_model.embed_query(hypothetical_doc)

    return hyde_vector

# --- Usage Example ---
# user_query = "What are the prigenerate_hyde_vectormary functions of the sympathetic nervous system?"
# query_vector = (user_query, my_llama_model, my_bge_m3_model)
#
# # Now you pass `query_vector` into your Vector DB for the similarity search!

## Initializing the BM25_Retriever

In [ ]:


# Extract all child chunks directly from Chroma
# Calling .get() provides a dictionary output containing the 'documents' and 'metadatas' arrays.
chroma_data = vectorstore.get()

# Reconstruct the LangChain Document objects so BM25 can read them
child_chunks = []
for i in range(len(chroma_data['documents'])):
    doc = Document(
        page_content=chroma_data['documents'][i],
        metadata=chroma_data['metadatas'][i] if chroma_data['metadatas'] else {}
    )
    child_chunks.append(doc)

print(f"Loaded {len(child_chunks)} child chunks directly from ChromaDB.")

# 3. Instantiate the BM25Retriever using the extracted chunks
my_bm25_retriever = BM25Retriever.from_documents(child_chunks)

## Define and Implementing the BM25 Search and Vector Search
    

In [ ]:
def retrieve_and_fuse(query: str, hyde_vector: list[float], bm25_retriever, vector_store, top_k: int = 5) -> list:
    """
    Executes a hybrid search and fuses the results using Reciprocal Rank Fusion (RRF).

    Args:
        query: The raw user string (for exact keyword matching).
        hyde_vector: The embedded hypothetical document (for semantic matching).
        bm25_retriever: Your initialized LangChain BM25Retriever.
        vector_store: Your initialized LangChain vector database.
        top_k: Number of initial documents to retrieve from each source.
    """

    # ---------------------------------------------------------
    # STEP 2: DUAL-STREAM RETRIEVAL
    # ---------------------------------------------------------

    # A. Keyword Search (BM25) - Uses the exact user query
    # We override the default k to ensure we get exactly top_k results
    bm25_retriever.k = top_k
    bm25_docs = bm25_retriever.invoke(query)

    # B. Vector Search - Uses the dense HyDE vector
    # similarity_search_by_vector allows us to bypass the text embedding step
    vector_docs = vector_store.similarity_search_by_vector(hyde_vector, k=top_k)

    print(f"Retrieved {len(bm25_docs)} BM25 docs and {len(vector_docs)} Vector docs.")

    # ---------------------------------------------------------
    # STEP 3: RECIPROCAL RANK FUSION (RRF)
    # ---------------------------------------------------------

    fused_scores = {}
    rrf_k = 60 # Standard constant used in the RRF formula

    def score_docs(docs):
        for rank, doc in enumerate(docs):
            # We use the chunk's text as the unique identifier.
            # (If you added a 'chunk_id' to your metadata during chunking, use doc.metadata['chunk_id'] instead!)
            doc_id = doc.page_content

            if doc_id not in fused_scores:
                fused_scores[doc_id] = {"doc": doc, "score": 0.0}

            # The RRF math: score = 1 / (rank + k)
            fused_scores[doc_id]["score"] += 1.0 / (rank + rrf_k)

    # Score both lists
    score_docs(bm25_docs)
    score_docs(vector_docs)

    # Sort the fused dictionary by the calculated RRF score in descending order
    reranked_results = sorted(fused_scores.values(), key=lambda x: x["score"], reverse=True)

    # Extract just the Document objects from the sorted list
    final_fused_docs = [item["doc"] for item in reranked_results]

    print(f"Successfully fused into {len(final_fused_docs)} unique child chunks.")

    return final_fused_docs

# --- Usage Example ---
# fused_child_chunks = retrieve_and_fuse(user_query, query_vector, my_bm25, my_vector_db)

## Implementing the cross-encoder re-ranker from BGE Ecosystem bge-reranker-large


In [ ]:

# 1. Initialize the Cross-Encoder model
# (Note: This will download the model the first time you run it. Switch to 'BAAI/bge-reranker-base' if 'large' is too heavy for your RAM limits)
bge_reranker = CrossEncoder("BAAI/bge-reranker-large", device="cuda")

def rerank_chunks(query: str, fused_docs: list, reranker_model, top_n: int = 3) -> list:
    """Passes the RRF fused docs through BGE-Reranker for precise scoring."""

    # The CrossEncoder expects a list of pairs: [[query, doc1], [query, doc2], ...]
    pairs = [[query, doc.page_content] for doc in fused_docs]

    # Predict similarity scores for each pair
    scores = reranker_model.predict(pairs)

    # Zip the scores with the documents and sort them highest to lowest
    scored_docs = list(zip(scores, fused_docs))
    scored_docs.sort(key=lambda x: x[0], reverse=True)

    # Return strictly the top N child chunks
    return [doc for score, doc in scored_docs[:top_n]]

# 2. Execute the re-ranking on your fused chunks!
# We pass in your 'query' variable and the 'fused_child_chunk' list you just generated
# top_child_chunks = rerank_chunks(query, fused_child_chunk, bge_reranker, top_n=4)

# print("\n--- Absolute Top Re-ranked Chunk ---")
# print(top_child_chunks[0].page_content)

## Fetch and Deduplicate the Parent Chunk


In [ ]:
parent_store = LocalFileStore(f"{current_dir}/parent_store_local/parent_store_local")


def fetch_and_deduplicate_parents(top_child_chunks: list, store) -> list:
    """Maps child chunks to parents and drops duplicates. Now includes page metadata."""
    unique_parent_ids = set()
    final_parent_docs = []

    for child in top_child_chunks:
        parent_id = child.metadata.get('doc_id')

        if parent_id and parent_id not in unique_parent_ids:
            unique_parent_ids.add(parent_id)

            parent_bytes = store.mget([parent_id])[0]

            if parent_bytes:
                parent_text = parent_bytes.decode('utf-8')
                # Carry forward the metadata from child → parent result
                final_parent_docs.append({
                    "text": parent_text,
                    "chapter": child.metadata.get("chapter", "Unknown"),
                    "section": child.metadata.get("section", "Unknown"),
                    "section_title": child.metadata.get("section-title", "Unknown"),
                    "page_start": child.metadata.get("page_start", "N/A"),
                    "page_end": child.metadata.get("page_end", "N/A"),
                })

    print(f"Reduced {len(top_child_chunks)} child chunks to {len(final_parent_docs)} unique parent chunks.")
    return final_parent_docs

In [ ]:
def encode_image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

## Generating the final answer

In [ ]:
def generate_final_answer(query: str, parent_docs: list, vision_llm_client):
    """Assembles text, metadata, and raw images for a citation-aware Vision model."""

    formatted_contexts = []

    for i, doc in enumerate(parent_docs):
        source_tag = f"[Source {i+1}]"
        meta_line = (
            f"Chapter: {doc['chapter']} | Section: {doc['section']} "
            f"| Title: {doc['section_title']} | Pages: {doc['page_start']}-{doc['page_end']}"
        )
        formatted_contexts.append(
            f"--- {source_tag} ({meta_line}) ---\n{doc['text']}\n"
        )

    combined_context = "\n".join(formatted_contexts)
    image_paths = re.findall(r'!\[[^\]]*\]\(([^)]+)\)', combined_context)

    json_schema = textwrap.dedent("""\
    {
      "answer": "Detailed answer string with inline citations like [Source 1].",
      "citations": [
        {
          "source_tag": "[Source 1]",
          "chapter": "Chapter name",
          "section": "Section number",
          "section_name": "Title of the section",
          "page_start": 1,
          "page_end": 2
        }
      ]
    }""")

    system_prompt = f"""You are an expert academic assistant. Your task is to answer the user's question based strictly on the provided textbook excerpts and any attached images.

### Rules:
1. **Rely Only on Context:** Base your answer entirely on the provided text and images. Do not use outside knowledge.
2. **Handle Missing Info:** If the provided context does not contain the answer, explicitly state: "The retrieved sections of the textbook do not contain enough information to answer this." Do not hallucinate.
3. **Synthesize Visuals:** If an image is provided alongside the text, analyze the raw image and reference its specific details to support your text-based answer.
4. **MANDATORY INLINE CITATIONS:** Every factual claim in your answer MUST be followed by the exact source tag indicating where you found the information (e.g., "Neurons transmit electrical signals [Source 1].").
5. **Structure:** Give all the answer in the given JSON Format.

### JSON Schema:
{json_schema}

### Textbook Context:
{combined_context}

### User Question:
{query}

Answer:"""

    message_content = [
        {"type": "text", "text": system_prompt}
    ]

    for path in image_paths:
        if os.path.exists(path):
            base64_image = encode_image_to_base64(path)
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
            })

    print(f"Passing context (with {len(parent_docs)} citations) and {len(image_paths)} images to Llama Vision...")
    response = vision_llm_client.invoke([HumanMessage(content=message_content)])
    return response.content

## Calling the All the above function to get the answer for the query

In [ ]:
def run_multimodal_rag_pipeline(
    query,
    llm,
    vision_llm,
    bge_m3_embedder,
    bm25_index,
    vector_db,
    bge_reranker,
    parent_retriever
):
    logger.info("🚀 Starting Multimodal RAG Pipeline")
    logger.info(f"🔎 Query: {query}")

    try:
        logger.info("1️⃣ Generating HyDE vector...")
        hyde_vec = generate_hyde_vector(query, llm, bge_m3_embedder)
        logger.info("✅ HyDE vector generated successfully")

        logger.info("2️⃣ Retrieving and fusing child chunks (BM25 + Vector)...")
        fused_children = retrieve_and_fuse(query, hyde_vec, bm25_index, vector_db)
        logger.info(f"✅ Retrieved & fused {len(fused_children)} child chunks")

        logger.info("3️⃣ Re-ranking fused chunks with Cross-Encoder...")
        top_children = rerank_chunks(query, fused_children, bge_reranker)
        logger.info(f"✅ Re-ranked top {len(top_children)} chunks")

        logger.info("4️⃣ Fetching and deduplicating Parent Chunks...")
        final_parents = fetch_and_deduplicate_parents(top_children, parent_retriever)
        logger.info(f"✅ Retrieved {len(final_parents)} unique parent chunks")

        logger.info("5️⃣ Generating final multimodal answer...")
        answer = generate_final_answer(query, final_parents, vision_llm)

        logger.info("🎉 Pipeline completed successfully")

        return answer

    except Exception as e:
        logger.exception("❌ Error occurred during RAG pipeline execution")
        raise

## Displaying it in Beutiful Way

In [ ]:
def display_rag_answer(result: dict, width: int = 100):
    """Pretty prints RAG answer + citations with page numbers."""

    print("\n" + "=" * width)
    print("📘 FINAL ANSWER".center(width))
    print("=" * width + "\n")

    wrapped_answer = textwrap.fill(result["answer"], width=width)
    print(wrapped_answer)

    print("\n" + "-" * width)
    print("📚 SOURCES".center(width))
    print("-" * width + "\n")

    for idx, citation in enumerate(result["citations"], 1):
        print(f"{idx}. {citation['source_tag']}")
        print(f"   📖 Chapter : {citation['chapter']}")
        print(f"   📑 Section : {citation['section']}")
        print(f"   🏷️ Title   : {citation['section_name']}")
        print(f"   📄 Pages   : {citation.get('page_start', 'N/A')} - {citation.get('page_end', 'N/A')}")
        print()

    print("=" * width + "\n")

In [ ]:

def give_final_answer(query): 
    final_answer = run_multimodal_rag_pipeline(query,llm,vision_llm,local_embeddings,my_bm25_retriever,vectorstore,bge_reranker,parent_store)
    result = json.loads(final_answer)
    return result

## Generating a Submission.csv File

In [ ]:
import json
import csv

# ---------- STEP 1: Load JSON file ----------
with open("./docs/queries.json", "r", encoding="utf-8") as f:
    queries = json.load(f)

total_queries = len(queries)
print(f"🚀 Starting processing of {total_queries} queries...\n")

# ---------- STEP 2: Create submission.csv ----------
with open("submission.csv", "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["query_id", "question", "answer", "citations"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for idx, item in enumerate(queries, start=1):
        query_id = item["query_id"]
        question = item["question"]

        print(f"⏳ Processing Query {idx}/{total_queries} (ID: {query_id})...")

        result = give_final_answer(question)

        answer_text = result["answer"]

        citation_texts = []
        for c in result["citations"]:
            citation_texts.append(
                f"{c['source_tag']} | Chapter: {c['chapter']} | "
                f"Section: {c['section']} | Title: {c['section_name']}"
            )

        citations_combined = " || ".join(citation_texts)

        writer.writerow({
            "query_id": query_id,
            "question": question,
            "answer": answer_text,
            "citations": citations_combined
        })

        print(f"✅ Query {idx} done\n")

print("🎉 submission.csv created successfully!")